# Large Anime Text Dataset — Exploratory Data Analysis

Quick tour of the export files in this repository. Assumes exports/ contains the release; if not, run python -m src.cli export first (or download the release assets).

`
pip install matplotlib pandas
`

In [ ]:
import json, os
import pandas as pd
import matplotlib.pyplot as plt

EX = "exports"
assert any(f.endswith(".jsonl") for f in os.listdir(EX)), "run: python -m src.cli export"

def load(name):
    return [json.loads(line) for line in open(f"{EX}/{name}.jsonl", encoding="utf-8") if line.strip()]

anime = load("anime")
print("anime rows:", len(anime))

In [ ]:
df = pd.DataFrame(anime)
score = pd.to_numeric(df["mean_score"], errors="coerce")
print(df[["title_canonical", "year", "type", "status"]].head(3))
print("\nmean_score summary:\n", score.describe())
print("\nmissing per field (top):\n", df.isna().mean().sort_values(ascending=False).head(8))

In [ ]:
y = df["year"].dropna().astype(int).value_counts().sort_index()
y.tail(12).plot(kind="bar", title="Works by year (tail)", color="#0f6b62")
plt.tight_layout(); plt.show()
print(y.tail(12))

In [ ]:
top_genres = pd.Series([g for g in df["genres"].explode() if isinstance(g, str)]).value_counts()
top_genres.head(15).plot(kind="barh", title="Top genres", color="#ff5a36")
plt.tight_layout(); plt.show()
print(top_genres.head(15))

In [ ]:
qa = load("qa")
inst = load("instructions")
rag = load("rag_chunks")
print("qa:", len(qa), "| instructions:", len(inst), "| rag_chunks:", len(rag))

import random
random.seed(42)
for r in random.sample(qa, 2):
    print("\nQ:", r["question"], "\nA:", r["answer"][:240])

In [ ]:
import statistics

def q(s):
    s = sorted(s); n = len(s)
    lo, med, hi = statistics.quantiles(s, n=4, method="inclusive")
    return {k: round(v, 1) for k, v in dict(min=min(s), q1=lo, median=med, q3=hi, max=max(s), mean=sum(s)/n).items()}

series = {
    "question chars": [len(r["question"]) for r in qa],
    "answer chars": [len(r["answer"]) for r in qa],
    "instruction chars": [len(r["instruction"]) for r in inst],
    "output chars": [len(r["output"]) for r in inst],
    "rag chunk chars": [len(r["text"]) for r in rag],
}
for label, s in series.items():
    print(f"{label:20s}", q(s))

## Next steps

- Deep-dive training recipes: docs/TRAINING.md.
- Terminal reference: docs/COMMANDS.md.
- Regenerate the dataset profile: python scripts/data_profile.py.
- Publish to the HF Hub (dry-run first): python scripts/upload_hf.py --repo <hf-username>/large-anime-text-dataset --dry-run.